# Calculating the forecast model AUC

In [5]:
from pathlib import Path
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
import xgboost as xgb


In [6]:
# ============================================================
# DATASET PATH
# ============================================================

data_set_path = str(
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset.csv"
)


In [7]:
# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                "is_churn contains numeric values "
                "other than 0/1:\n"
                f"{values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            "Unknown values in is_churn:\n"
            f"{unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE HISTORICAL TRAINING DATA
# ============================================================

def prepare_data(
    data_set_path,
    ext="_groupscore",
    as_retention=True
):

    score_path = (
        data_set_path.replace(
            ".csv",
            f"{ext}.csv"
        )
    )

    if not os.path.isfile(
        score_path
    ):
        raise FileNotFoundError(
            "Run Listing 6.3 / 8.1 first.\n"
            f"Expected:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:
        raise ValueError(
            '"is_churn" column missing.'
        )

    is_churn = convert_churn_target(
        grouped_data["is_churn"]
    )

    if as_retention:

        # False = churn
        # True = retention
        y = ~is_churn

    else:

        y = is_churn

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad_cols = (
            X.columns[
                X.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            "Missing/non-numeric model features:\n"
            f"{bad_cols}"
        )

    return X, y


# ============================================================
# RELOAD LOGISTIC REGRESSION MODEL
# ============================================================

def reload_regression(
    data_set_path
):

    pickle_path = (
        data_set_path.replace(
            ".csv",
            "_logreg_model.pkl"
        )
    )

    if not os.path.isfile(
        pickle_path
    ):
        raise FileNotFoundError(
            "Run Listing 8.2 first.\n"
            f"Expected model:\n"
            f"{pickle_path}"
        )

    with open(
        pickle_path,
        "rb"
    ) as fid:

        saved_object = pickle.load(
            fid
        )

    # Our Listing 8.2 stored:
    #
    # {
    #     "model": ...,
    #     "feature_names": [...],
    #     "target": "retention"
    # }

    if isinstance(
        saved_object,
        dict
    ):

        if "model" not in saved_object:
            raise ValueError(
                "Saved model package does not "
                "contain a model."
            )

        logreg_model = (
            saved_object["model"]
        )

        feature_names = (
            saved_object.get(
                "feature_names"
            )
        )

        target = (
            saved_object.get(
                "target",
                "retention"
            )
        )

    else:

        # Compatibility with book-style
        # raw sklearn pickle
        logreg_model = saved_object
        feature_names = None
        target = "retention"

    return (
        logreg_model,
        feature_names,
        target
    )


# ============================================================
# LISTING 9.1 — REGRESSION AUC
# ============================================================

def regression_auc(
    data_set_path
):

    # --------------------------------------------------------
    # Load fitted model
    # --------------------------------------------------------

    (
        logreg_model,
        feature_names,
        target
    ) = reload_regression(
        data_set_path
    )

    # --------------------------------------------------------
    # Load training observations
    # --------------------------------------------------------

    X, y = prepare_data(
        data_set_path,
        ext="_groupscore",
        as_retention=True
    )

    # --------------------------------------------------------
    # Preserve exact model feature order
    # --------------------------------------------------------

    if feature_names is not None:

        missing_features = [
            col
            for col in feature_names
            if col not in X.columns
        ]

        if missing_features:
            raise ValueError(
                "Training dataset is missing "
                "model features:\n"
                f"{missing_features}"
            )

        X = X[
            feature_names
        ]

    # --------------------------------------------------------
    # Predict probabilities
    # --------------------------------------------------------

    predictions = (
        logreg_model.predict_proba(
            X
        )
    )

    print(
        "Model classes:",
        logreg_model.classes_
    )

    # --------------------------------------------------------
    # Positive class here is retention=True
    # --------------------------------------------------------

    class_to_index = {
        class_value: i
        for i, class_value
        in enumerate(
            logreg_model.classes_
        )
    }

    if True not in class_to_index:
        raise ValueError(
            "Retention=True class not found."
        )

    retain_index = (
        class_to_index[True]
    )

    retain_probability = (
        predictions[
            :,
            retain_index
        ]
    )

    # --------------------------------------------------------
    # AUC
    # --------------------------------------------------------

    auc_score = roc_auc_score(
        y,
        retain_probability
    )

    print(
        f"\nRegression AUC score = "
        f"{auc_score:.4f}"
    )

    print(
        f"Observations = {len(y):,}"
    )

    print(
        f"Features = {X.shape[1]}"
    )

    print(
        f"Retention observations = "
        f"{y.sum():,}"
    )

    print(
        f"Churn observations = "
        f"{(~y).sum():,}"
    )

    return auc_score


# ============================================================
# RUN LISTING 9.1
# ============================================================

auc_score = regression_auc(
    data_set_path
)

Model classes: [False  True]

Regression AUC score = 0.6444
Observations = 32,897
Features = 2
Retention observations = 32,252
Churn observations = 645


# Calculating the forecast model lift

In [8]:
# ============================================================
# ROBUST TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected numeric churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE HISTORICAL DATA
# ============================================================

def prepare_data(
    data_set_path,
    ext="_groupscore",
    as_retention=True
):

    score_path = (
        data_set_path.replace(
            ".csv",
            f"{ext}.csv"
        )
    )

    if not os.path.isfile(score_path):
        raise FileNotFoundError(
            "Run Listing 6.3 / 8.1 first.\n"
            f"Expected:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:
        raise ValueError(
            '"is_churn" column is missing.'
        )

    is_churn = convert_churn_target(
        grouped_data["is_churn"]
    )

    if as_retention:
        y = ~is_churn
    else:
        y = is_churn

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad_cols = (
            X.columns[
                X.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            "NaN/non-numeric features found in:\n"
            f"{bad_cols}"
        )

    return X, y


# ============================================================
# RELOAD MODEL
# ============================================================

def reload_regression(
    data_set_path
):

    pickle_path = (
        data_set_path.replace(
            ".csv",
            "_logreg_model.pkl"
        )
    )

    if not os.path.isfile(
        pickle_path
    ):
        raise FileNotFoundError(
            "Run Listing 8.2 first.\n"
            f"Expected model:\n{pickle_path}"
        )

    with open(
        pickle_path,
        "rb"
    ) as fid:

        saved_object = pickle.load(
            fid
        )

    if isinstance(
        saved_object,
        dict
    ):

        logreg_model = (
            saved_object["model"]
        )

        feature_names = (
            saved_object.get(
                "feature_names"
            )
        )

    else:

        logreg_model = saved_object
        feature_names = None

    return (
        logreg_model,
        feature_names
    )


# ============================================================
# CALCULATE TOP-DECILE LIFT
# ============================================================

def calc_lift(
    y_true,
    y_pred,
    top_fraction=0.10
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred must have the same length."
        )

    if len(y_true) == 0:
        raise ValueError(
            "No observations supplied."
        )

    if not 0 < top_fraction < 1:
        raise ValueError(
            "top_fraction must be between 0 and 1."
        )

    if np.unique(y_pred).size < 10:
        print(
            "Warning: fewer than 10 unique prediction values."
        )

    # --------------------------------------------------------
    # Overall churn rate
    # --------------------------------------------------------

    overall_churn = (
        y_true.mean()
    )

    if overall_churn == 0:
        raise ValueError(
            "Overall churn rate is zero, so lift is undefined."
        )

    # --------------------------------------------------------
    # Rank highest predicted churn first
    # --------------------------------------------------------

    order = np.argsort(
        y_pred
    )[::-1]

    y_sorted = y_true[
        order
    ]

    pred_sorted = y_pred[
        order
    ]

    # --------------------------------------------------------
    # Top 10%
    # --------------------------------------------------------

    top_n = max(
        1,
        int(
            np.ceil(
                len(y_true)
                * top_fraction
            )
        )
    )

    top_y = y_sorted[
        :top_n
    ]

    top_predictions = pred_sorted[
        :top_n
    ]

    top_decile_churn = (
        top_y.mean()
    )

    lift = (
        top_decile_churn
        / overall_churn
    )

    return {
        "lift": lift,
        "overall_churn_rate":
            overall_churn,
        "top_decile_churn_rate":
            top_decile_churn,
        "top_decile_n":
            top_n,
        "top_decile_churn_count":
            int(top_y.sum()),
        "min_prediction_in_top_decile":
            float(
                top_predictions.min()
            )
    }


# ============================================================
# LISTING 9.2 — TOP DECILE LIFT
# ============================================================

def top_decile_lift(
    data_set_path
):

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    (
        logreg_model,
        feature_names
    ) = reload_regression(
        data_set_path
    )

    # --------------------------------------------------------
    # Important:
    # y=True now means CHURN
    # --------------------------------------------------------

    X, y = prepare_data(
        data_set_path,
        ext="_groupscore",
        as_retention=False
    )

    # --------------------------------------------------------
    # Exact model feature order
    # --------------------------------------------------------

    if feature_names is not None:

        missing_features = [
            feature
            for feature in feature_names
            if feature not in X.columns
        ]

        if missing_features:
            raise ValueError(
                "Dataset is missing model features:\n"
                f"{missing_features}"
            )

        X = X[
            feature_names
        ]

    # --------------------------------------------------------
    # Model itself was trained on RETENTION:
    #
    # False = churn
    # True  = retention
    # --------------------------------------------------------

    predictions = (
        logreg_model.predict_proba(
            X
        )
    )

    print(
        "Model classes:",
        logreg_model.classes_
    )

    class_to_index = {
        class_value: i
        for i, class_value
        in enumerate(
            logreg_model.classes_
        )
    }

    if False not in class_to_index:
        raise ValueError(
            "Churn=False model class not found."
        )

    churn_index = (
        class_to_index[False]
    )

    churn_prob = (
        predictions[
            :,
            churn_index
        ]
    )

    # --------------------------------------------------------
    # Calculate lift
    # --------------------------------------------------------

    result = calc_lift(
        y_true=y,
        y_pred=churn_prob,
        top_fraction=0.10
    )

    print(
        f"\nRegression Lift score = "
        f"{result['lift']:.3f}"
    )

    print(
        f"Overall churn rate = "
        f"{result['overall_churn_rate']:.3%}"
    )

    print(
        f"Top-decile churn rate = "
        f"{result['top_decile_churn_rate']:.3%}"
    )

    print(
        f"Accounts in top decile = "
        f"{result['top_decile_n']:,}"
    )

    print(
        f"Churns in top decile = "
        f"{result['top_decile_churn_count']:,}"
    )

    print(
        f"Minimum predicted churn probability "
        f"in top decile = "
        f"{result['min_prediction_in_top_decile']:.4f}"
    )

    return result


# ============================================================
# RUN LISTING 9.2
# ============================================================

lift_result = top_decile_lift(
    data_set_path
)

Model classes: [False  True]

Regression Lift score = 2.744
Overall churn rate = 1.961%
Top-decile churn rate = 5.380%
Accounts in top decile = 3,290
Churns in top decile = 177
Minimum predicted churn probability in top decile = 0.0340


# Backtesting with Python time-series cross-validation

In [9]:
# ============================================================
# TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected numeric churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# LOAD GROUPED DATA
# ============================================================

def prepare_data(
    data_set_path
):

    score_path = (
        data_set_path.replace(
            ".csv",
            "_groupscore.csv"
        )
    )

    if not os.path.isfile(score_path):
        raise FileNotFoundError(
            f"Missing:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:
        raise ValueError(
            "is_churn column missing."
        )

    # --------------------------------------------
    # Churn is positive class
    # --------------------------------------------

    y = convert_churn_target(
        grouped_data["is_churn"]
    )

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            f"NaN/non-numeric features: {bad}"
        )

    # --------------------------------------------
    # Extract observation date from second index
    # --------------------------------------------

    dates = pd.to_datetime(
        X.index.get_level_values(1),
        errors="raise"
    )

    return X, y, dates


# ============================================================
# TOP-DECILE LIFT
# ============================================================

def calc_lift(
    y_true,
    y_pred,
    top_fraction=0.10
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    if len(y_true) == 0:
        return np.nan

    overall_churn = (
        y_true.mean()
    )

    if overall_churn == 0:
        return np.nan

    order = np.argsort(
        y_pred
    )[::-1]

    y_sorted = y_true[
        order
    ]

    top_n = max(
        1,
        int(
            np.ceil(
                len(y_true)
                * top_fraction
            )
        )
    )

    top_churn_rate = (
        y_sorted[
            :top_n
        ]
        .mean()
    )

    return (
        top_churn_rate
        / overall_churn
    )


# ============================================================
# DATE-BASED EXPANDING BACKTEST
# ============================================================

def backtest(
    data_set_path,
    n_test_split=5
):

    X, y, dates = prepare_data(
        data_set_path
    )

    # ========================================================
    # SORT BY OBSERVATION DATE
    # ========================================================

    order = np.argsort(
        dates.values,
        kind="stable"
    )

    X = X.iloc[
        order
    ].copy()

    y = y.iloc[
        order
    ].copy()

    dates = dates[
        order
    ]


    # ========================================================
    # BASIC DATA SUMMARY
    # ========================================================

    unique_dates = np.array(
        sorted(
            pd.unique(dates)
        )
    )

    print(
        f"Observations: {len(X):,}"
    )

    print(
        f"Features: {X.shape[1]}"
    )

    print(
        f"Observation dates: {len(unique_dates)}"
    )

    print(
        f"First observation date: "
        f"{pd.Timestamp(unique_dates[0]).date()}"
    )

    print(
        f"Last observation date: "
        f"{pd.Timestamp(unique_dates[-1]).date()}"
    )

    print(
        f"Overall churn rate: "
        f"{y.mean():.3%}"
    )


    # ========================================================
    # SHOW CHURN BY DATE
    # ========================================================

    date_summary = pd.DataFrame({
        "date": dates,
        "is_churn": y.to_numpy()
    })

    date_summary = (
        date_summary
        .groupby("date")
        .agg(
            observations=(
                "is_churn",
                "size"
            ),
            churns=(
                "is_churn",
                "sum"
            ),
            churn_rate=(
                "is_churn",
                "mean"
            )
        )
        .reset_index()
    )

    print(
        "\nChurn by observation date:"
    )

    display(
        date_summary
    )


    # ========================================================
    # CREATE WHOLE-DATE TEST BLOCKS
    # ========================================================

    if len(unique_dates) < 3:
        raise ValueError(
            "Not enough unique observation dates "
            "for temporal backtesting."
        )

    # Leave initial dates for training.
    # Split remaining dates into approximately equal test blocks.
    test_date_blocks = np.array_split(
        unique_dates[1:],
        n_test_split
    )


    # ========================================================
    # RUN EXPANDING-WINDOW BACKTEST
    # ========================================================

    fold_results = []

    for fold_number, test_dates in enumerate(
        test_date_blocks,
        start=1
    ):

        if len(test_dates) == 0:
            continue

        test_start = pd.Timestamp(
            test_dates[0]
        )

        test_end = pd.Timestamp(
            test_dates[-1]
        )

        # --------------------------------------------
        # Training = everything before test start
        # --------------------------------------------

        train_mask = (
            dates < test_start
        )

        test_mask = (
            (dates >= test_start)
            &
            (dates <= test_end)
        )

        X_train = X.loc[
            train_mask
        ]

        y_train = y.loc[
            train_mask
        ]

        X_test = X.loc[
            test_mask
        ]

        y_test = y.loc[
            test_mask
        ]


        # ====================================================
        # CLASS CHECK
        # ====================================================

        train_classes = (
            y_train
            .value_counts()
            .to_dict()
        )

        test_classes = (
            y_test
            .value_counts()
            .to_dict()
        )

        print(
            f"\nFold {fold_number}"
        )

        print(
            f"Train: "
            f"{dates[train_mask].min().date()} "
            f"to "
            f"{dates[train_mask].max().date()}"
        )

        print(
            f"Test:  "
            f"{test_start.date()} "
            f"to "
            f"{test_end.date()}"
        )

        print(
            f"Train classes: {train_classes}"
        )

        print(
            f"Test classes: {test_classes}"
        )


        # ====================================================
        # CANNOT FIT LOGISTIC REGRESSION WITH ONE CLASS
        # ====================================================

        if y_train.nunique() < 2:

            print(
                "Skipping fold: training data "
                "contains only one class."
            )

            fold_results.append(
                {
                    "fold": fold_number,
                    "status": "skipped_train_one_class",
                    "train_start":
                        dates[train_mask].min(),
                    "train_end":
                        dates[train_mask].max(),
                    "test_start":
                        test_start,
                    "test_end":
                        test_end,
                    "train_n":
                        len(X_train),
                    "test_n":
                        len(X_test),
                    "train_churns":
                        int(y_train.sum()),
                    "test_churns":
                        int(y_test.sum()),
                    "train_churn_rate":
                        y_train.mean(),
                    "test_churn_rate":
                        y_test.mean(),
                    "AUC":
                        np.nan,
                    "lift":
                        np.nan
                }
            )

            continue


        # ====================================================
        # MODEL
        # ====================================================

        # L1 logistic regression.
        #
        # saga avoids the newer sklearn liblinear
        # penalty deprecation warnings.
        churn_reg = LogisticRegression(
            solver="saga",
            l1_ratio=1.0,
            C=1.0,
            fit_intercept=True,
            max_iter=5000,
            random_state=42
        )

        churn_reg.fit(
            X_train,
            y_train
        )


        # ====================================================
        # PREDICT CHURN
        # ====================================================

        class_to_index = {
            class_value: i
            for i, class_value
            in enumerate(
                churn_reg.classes_
            )
        }

        churn_index = (
            class_to_index[True]
        )

        churn_prob = (
            churn_reg
            .predict_proba(
                X_test
            )[
                :,
                churn_index
            ]
        )


        # ====================================================
        # AUC
        # ====================================================

        if y_test.nunique() == 2:

            auc = roc_auc_score(
                y_test,
                churn_prob
            )

        else:

            # AUC undefined if test fold has
            # only churn or only retention.
            auc = np.nan


        # ====================================================
        # LIFT
        # ====================================================

        lift = calc_lift(
            y_test,
            churn_prob
        )


        # ====================================================
        # SAVE FOLD
        # ====================================================

        fold_results.append(
            {
                "fold":
                    fold_number,

                "status":
                    "ok",

                "train_start":
                    dates[
                        train_mask
                    ].min(),

                "train_end":
                    dates[
                        train_mask
                    ].max(),

                "test_start":
                    test_start,

                "test_end":
                    test_end,

                "train_n":
                    len(X_train),

                "test_n":
                    len(X_test),

                "train_churns":
                    int(
                        y_train.sum()
                    ),

                "test_churns":
                    int(
                        y_test.sum()
                    ),

                "train_churn_rate":
                    y_train.mean(),

                "test_churn_rate":
                    y_test.mean(),

                "AUC":
                    auc,

                "lift":
                    lift
            }
        )


    # ========================================================
    # RESULTS
    # ========================================================

    fold_df = pd.DataFrame(
        fold_results
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            "_backtest.csv"
        )
    )

    fold_df.to_csv(
        save_path,
        index=False
    )

    print(
        f"\nSaved backtest to:\n"
        f"{save_path}"
    )

    print(
        "\nBacktest results:"
    )

    display(
        fold_df
    )


    # ========================================================
    # VALID FOLDS ONLY
    # ========================================================

    valid = fold_df[
        fold_df["status"] == "ok"
    ]

    if len(valid) > 0:

        print(
            f"\nValid folds: "
            f"{len(valid)}"
        )

        print(
            f"Mean test AUC: "
            f"{valid['AUC'].mean():.4f}"
        )

        print(
            f"Mean top-decile lift: "
            f"{valid['lift'].mean():.4f}"
        )

        print(
            f"AUC std: "
            f"{valid['AUC'].std():.4f}"
        )

        print(
            f"Lift std: "
            f"{valid['lift'].std():.4f}"
        )

    else:

        print(
            "\nNo valid temporal folds could be fitted."
        )

    return (
        fold_df,
        date_summary
    )


# ============================================================
# RUN LISTING 9.3
# ============================================================

backtest_results, churn_by_date = backtest(
    data_set_path=data_set_path,
    n_test_split=5
)

Observations: 32,897
Features: 2
Observation dates: 92
First observation date: 2020-02-09
Last observation date: 2020-05-10
Overall churn rate: 1.961%

Churn by observation date:


,date,observations,churns,churn_rate
0,2020-02-09,306,0,0.0
1,2020-02-10,289,0,0.0
2,2020-02-11,281,0,0.0
3,2020-02-12,316,0,0.0
4,2020-02-13,269,0,0.0
...,...,...,...,...
87,2020-05-06,395,0,0.0
88,2020-05-07,398,0,0.0
89,2020-05-08,382,0,0.0
90,2020-05-09,419,0,0.0



Fold 1
Train: 2020-02-09 to 2020-02-09
Test:  2020-02-10 to 2020-02-28
Train classes: {False: 306}
Test classes: {False: 6459, True: 48}
Skipping fold: training data contains only one class.

Fold 2
Train: 2020-02-09 to 2020-02-28
Test:  2020-02-29 to 2020-03-17
Train classes: {False: 6765, True: 48}
Test classes: {False: 5777, True: 193}

Fold 3
Train: 2020-02-09 to 2020-03-17
Test:  2020-03-18 to 2020-04-04
Train classes: {False: 12542, True: 241}
Test classes: {False: 6154, True: 164}

Fold 4
Train: 2020-02-09 to 2020-04-04
Test:  2020-04-05 to 2020-04-22
Train classes: {False: 18696, True: 405}
Test classes: {False: 6301, True: 153}

Fold 5
Train: 2020-02-09 to 2020-04-22
Test:  2020-04-23 to 2020-05-10
Train classes: {False: 24997, True: 558}
Test classes: {False: 7255, True: 87}

Saved backtest to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_backtest.csv

Backtest results:


,fold,status,train_start,train_end,test_start,test_end,train_n,test_n,train_churns,test_churns,train_churn_rate,test_churn_rate,AUC,lift
0,1,skipped_train_one_class,2020-02-09,2020-02-09,2020-02-10,2020-02-28,306,6507,0,48,0.000000,0.007377,NaN,NaN
1,2,ok,2020-02-09,2020-02-28,2020-02-29,2020-03-17,6813,5970,48,193,0.007045,0.032328,0.548838,2.176166
2,3,ok,2020-02-09,2020-03-17,2020-03-18,2020-04-04,12783,6318,241,164,0.018853,0.025958,0.463670,1.036257
3,4,ok,2020-02-09,2020-04-04,2020-04-05,2020-04-22,19101,6454,405,153,0.021203,0.023706,0.655120,2.677250
4,5,ok,2020-02-09,2020-04-22,2020-04-23,2020-05-10,25555,7342,558,87,0.021835,0.011850,0.609202,2.296348



Valid folds: 4
Mean test AUC: 0.5692
Mean top-decile lift: 2.0465
AUC std: 0.0827
Lift std: 0.7066


# Regression using the control parameter C

In [10]:
# ============================================================
# TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE DATA
# ============================================================

def prepare_data(
    data_set_path,
    ext="_groupscore",
    as_retention=True
):

    score_path = data_set_path.replace(
        ".csv",
        f"{ext}.csv"
    )

    if not os.path.isfile(score_path):
        raise FileNotFoundError(
            "Grouped score dataset not found.\n"
            f"Expected:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:
        raise ValueError(
            '"is_churn" column is missing.'
        )

    is_churn = convert_churn_target(
        grouped_data["is_churn"]
    )

    if as_retention:
        y = ~is_churn
    else:
        y = is_churn

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            "NaN/non-numeric features found in:\n"
            f"{bad_columns}"
        )

    return X, y


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    x = np.asarray(
        x,
        dtype=float
    )

    x = np.clip(
        x,
        -709,
        709
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


# ============================================================
# CALCULATE IMPACTS
# ============================================================

def calculate_impacts(
    retain_reg
):

    intercept = float(
        retain_reg.intercept_[0]
    )

    coefficients = (
        retain_reg.coef_[0]
    )

    average_retain = float(
        sigmoid(intercept)
    )

    one_stdev_retains = sigmoid(
        intercept + coefficients
    )

    one_stdev_impact = (
        one_stdev_retains
        - average_retain
    )

    return (
        one_stdev_impact,
        average_retain
    )


# ============================================================
# LOAD GROUP DESCRIPTIONS
# ============================================================

def load_group_descriptions(
    data_set_path,
    feature_names
):

    group_path = data_set_path.replace(
        ".csv",
        "_groupmets.csv"
    )

    if not os.path.isfile(group_path):

        return {
            feature: feature
            for feature in feature_names
        }

    group_df = pd.read_csv(
        group_path,
        index_col=0
    )

    if "group" not in group_df.columns:

        return {
            feature: feature
            for feature in feature_names
        }

    descriptions = {}

    for feature in feature_names:

        if not feature.startswith("group_"):

            descriptions[feature] = feature
            continue

        try:

            group_number = int(
                feature.replace(
                    "group_",
                    ""
                )
            )

        except ValueError:

            descriptions[feature] = feature
            continue

        members = (
            group_df.index[
                group_df["group"] == group_number
            ]
            .tolist()
        )

        descriptions[feature] = (
            ", ".join(
                map(str, members)
            )
            if members
            else "(no metrics found)"
        )

    return descriptions


# ============================================================
# SAVE REGRESSION SUMMARY
# ============================================================

def save_regression_summary(
    data_set_path,
    retain_reg,
    feature_names,
    ext=""
):

    (
        one_stdev_impact,
        average_retain
    ) = calculate_impacts(
        retain_reg
    )

    descriptions = load_group_descriptions(
        data_set_path,
        feature_names
    )

    rows = []

    for (
        feature,
        coefficient,
        impact
    ) in zip(
        feature_names,
        retain_reg.coef_[0],
        one_stdev_impact
    ):

        rows.append(
            {
                "group_metric_offset":
                    feature,

                "weight":
                    coefficient,

                "retain_impact":
                    impact,

                "group_metrics":
                    descriptions.get(
                        feature,
                        feature
                    )
            }
        )

    rows.append(
        {
            "group_metric_offset":
                "offset",

            "weight":
                float(
                    retain_reg.intercept_[0]
                ),

            "retain_impact":
                average_retain,

            "group_metrics":
                "(baseline)"
        }
    )

    coef_df = pd.DataFrame(
        rows
    )

    save_path = data_set_path.replace(
        ".csv",
        f"_logreg_summary{ext}.csv"
    )

    coef_df.to_csv(
        save_path,
        index=False
    )

    print(
        f"\nSaved coefficients to:\n"
        f"{save_path}"
    )

    return coef_df


# ============================================================
# SAVE MODEL
# ============================================================

def save_regression_model(
    data_set_path,
    retain_reg,
    feature_names,
    ext=""
):

    pickle_path = data_set_path.replace(
        ".csv",
        f"_logreg_model{ext}.pkl"
    )

    model_package = {
        "model":
            retain_reg,

        "feature_names":
            list(feature_names),

        "target":
            "retention"
    }

    with open(
        pickle_path,
        "wb"
    ) as fid:

        pickle.dump(
            model_package,
            fid
        )

    print(
        f"\nSaved model to:\n"
        f"{pickle_path}"
    )

    return pickle_path


# ============================================================
# SAVE PREDICTIONS
# ============================================================

def save_dataset_predictions(
    data_set_path,
    retain_reg,
    X,
    ext=""
):

    probabilities = (
        retain_reg.predict_proba(
            X
        )
    )

    class_to_index = {
        class_value: i
        for i, class_value
        in enumerate(
            retain_reg.classes_
        )
    }

    if False not in class_to_index:
        raise ValueError(
            "Expected churn class False not found."
        )

    if True not in class_to_index:
        raise ValueError(
            "Expected retention class True not found."
        )

    churn_index = (
        class_to_index[False]
    )

    retain_index = (
        class_to_index[True]
    )

    predict_df = pd.DataFrame(
        {
            "churn_prob":
                probabilities[
                    :,
                    churn_index
                ],

            "retain_prob":
                probabilities[
                    :,
                    retain_index
                ]
        },
        index=X.index
    )

    predict_path = data_set_path.replace(
        ".csv",
        f"_predictions{ext}.csv"
    )

    predict_df.to_csv(
        predict_path,
        header=True
    )

    print(
        f"\nSaved predictions to:\n"
        f"{predict_path}"
    )

    return predict_df


# ============================================================
# REGRESSION USING C PARAMETER
# ============================================================

def regression_cparam(
    data_set_path,
    C_param
):

    if C_param <= 0:
        raise ValueError(
            "C_param must be greater than 0."
        )

    print(
        "\nLoading grouped training data..."
    )

    X, y = prepare_data(
        data_set_path,
        ext="_groupscore",
        as_retention=True
    )

    print(
        f"Rows: {len(X):,}"
    )

    print(
        f"Features: {X.shape[1]}"
    )

    print(
        f"Churn observations: {(~y).sum():,}"
    )

    print(
        f"Retention observations: {y.sum():,}"
    )

    print(
        f"\nTraining logistic regression "
        f"with C = {C_param:.3f}..."
    )

    # ========================================================
    # L1 LOGISTIC REGRESSION
    # ========================================================

    retain_reg = LogisticRegression(
        C=C_param,
        solver="saga",
        l1_ratio=1.0,
        fit_intercept=True,
        max_iter=5000,
        random_state=42
    )

    retain_reg.fit(
        X,
        y
    )

    # ========================================================
    # FILE SUFFIX
    # ========================================================

    c_ext = (
        f"_c{C_param:.3f}"
    )

    # ========================================================
    # SAVE SUMMARY
    # ========================================================

    summary = save_regression_summary(
        data_set_path,
        retain_reg,
        feature_names=X.columns,
        ext=c_ext
    )

    # ========================================================
    # SAVE MODEL
    # ========================================================

    save_regression_model(
        data_set_path,
        retain_reg,
        feature_names=X.columns,
        ext=c_ext
    )

    # ========================================================
    # SAVE PREDICTIONS
    # ========================================================

    predictions = save_dataset_predictions(
        data_set_path,
        retain_reg,
        X,
        ext=c_ext
    )

    # ========================================================
    # NON-ZERO COEFFICIENTS
    # ========================================================

    n_nonzero = int(
        np.count_nonzero(
            retain_reg.coef_[0]
        )
    )

    print(
        f"\nC = {C_param:.3f}"
    )

    print(
        f"Non-zero coefficients = "
        f"{n_nonzero} / {X.shape[1]}"
    )

    print(
        "\nCoefficients:"
    )

    coefficient_df = pd.DataFrame(
        {
            "feature":
                X.columns,

            "coefficient":
                retain_reg.coef_[0]
        }
    )

    display(
        coefficient_df
    )

    return (
        retain_reg,
        summary,
        predictions
    )


# ============================================================
# RUN EXAMPLE
# ============================================================

(
    retain_reg_c,
    regression_summary_c,
    predictions_c
) = regression_cparam(
    data_set_path=data_set_path,
    C_param=1.0
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print(
    "\nRegression summary:"
)

display(
    regression_summary_c
)


print(
    "\nFirst predictions:"
)

display(
    predictions_c.head()
)


Loading grouped training data...
Rows: 32,897
Features: 2
Churn observations: 645
Retention observations: 32,252

Training logistic regression with C = 1.000...

Saved coefficients to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_logreg_summary_c1.000.csv

Saved model to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_logreg_model_c1.000.pkl

Saved predictions to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_predictions_c1.000.csv

C = 1.000
Non-zero coefficients = 2 / 2

Coefficients:


,feature,coefficient
0,group_1,0.515799
1,group_2,0.210156



Regression summary:


,group_metric_offset,weight,retain_impact,group_metrics
0,group_1,0.515799,0.006729,like_per_month
1,group_2,0.210156,0.003153,account_tenure
2,offset,4.065277,0.983131,(baseline)



First predictions:


,,churn_prob,retain_prob
account_id,observation_date,,
82,2020-02-09,0.016758,0.983242
122,2020-02-09,0.044442,0.955558
184,2020-02-09,0.038414,0.961586
262,2020-02-09,0.030588,0.969412
291,2020-02-09,0.033019,0.966981


# Regression C parameter cross-validation 

In [13]:
#===================================================
# TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            raise ValueError(
                f"Unexpected churn values: {values}"
            )

        return (
            series
            .astype(int)
            .astype(bool)
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = (
        set(cleaned.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return (
        cleaned
        .map(mapping)
        .astype(bool)
    )


# ============================================================
# PREPARE GROUPED DATA
# ============================================================

def prepare_data(
    data_set_path
):

    score_path = data_set_path.replace(
        ".csv",
        "_groupscore.csv"
    )

    if not os.path.isfile(score_path):

        raise FileNotFoundError(
            "Grouped score dataset not found.\n"
            f"Expected:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:

        raise ValueError(
            '"is_churn" column is missing.'
        )

    # True = churn
    y = convert_churn_target(
        grouped_data["is_churn"]
    )

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad_columns = (
            X.columns[
                X.isna().any()
            ]
            .tolist()
        )

        raise ValueError(
            "NaN/non-numeric features found in:\n"
            f"{bad_columns}"
        )

    dates = pd.to_datetime(
        X.index.get_level_values(1),
        errors="raise"
    )

    # Sort chronologically
    order = np.argsort(
        dates.values,
        kind="stable"
    )

    X = X.iloc[
        order
    ].copy()

    y = y.iloc[
        order
    ].copy()

    dates = dates[
        order
    ]

    return X, y, dates


# ============================================================
# TOP-DECILE LIFT
# ============================================================

def calc_lift(
    y_true,
    y_pred,
    top_fraction=0.10
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    if len(y_true) == 0:
        return np.nan

    overall_churn = (
        y_true.mean()
    )

    if overall_churn <= 0:
        return np.nan

    order = np.argsort(
        y_pred
    )[::-1]

    y_sorted = y_true[
        order
    ]

    top_n = max(
        1,
        int(
            np.ceil(
                len(y_true)
                * top_fraction
            )
        )
    )

    top_churn_rate = (
        y_sorted[
            :top_n
        ]
        .mean()
    )

    return (
        top_churn_rate
        / overall_churn
    )


# ============================================================
# CREATE DATE-BASED EXPANDING SPLITS
# ============================================================

def create_date_splits(
    dates,
    n_test_split
):

    unique_dates = np.array(
        sorted(
            pd.unique(dates)
        )
    )

    if len(unique_dates) < 3:

        raise ValueError(
            "Not enough unique observation dates."
        )

    if n_test_split >= len(unique_dates):

        raise ValueError(
            "n_test_split must be smaller than "
            "the number of unique observation dates."
        )

    # Keep at least the first date for initial training.
    test_blocks = np.array_split(
        unique_dates[1:],
        n_test_split
    )

    splits = []

    for fold_number, test_dates in enumerate(
        test_blocks,
        start=1
    ):

        if len(test_dates) == 0:
            continue

        test_start = pd.Timestamp(
            test_dates[0]
        )

        test_end = pd.Timestamp(
            test_dates[-1]
        )

        train_mask = (
            dates < test_start
        )

        test_mask = (
            (dates >= test_start)
            &
            (dates <= test_end)
        )

        train_idx = np.where(
            train_mask
        )[0]

        test_idx = np.where(
            test_mask
        )[0]

        splits.append(
            (
                fold_number,
                train_idx,
                test_idx,
                test_start,
                test_end
            )
        )

    return splits


# ============================================================
# COUNT NON-ZERO WEIGHTS FOR EACH C
# ============================================================

def test_n_weights(
    X,
    y,
    C_values
):

    results = []

    for c_value in C_values:

        # Need both classes
        if y.nunique() < 2:

            n_weights = np.nan

        else:

            model = LogisticRegression(
                C=c_value,
                solver="saga",
                l1_ratio=1.0,
                fit_intercept=True,
                max_iter=5000,
                random_state=42
            )

            model.fit(
                X,
                y
            )

            n_weights = int(
                np.count_nonzero(
                    model.coef_[0]
                )
            )

        results.append(
            {
                "C":
                    c_value,

                "n_weight":
                    n_weights
            }
        )

    return pd.DataFrame(
        results
    )


# ============================================================
# PLOT CROSS-VALIDATION RESULTS
# ============================================================

def plot_regression_test(
    data_set_path,
    result_df
):

    plot_df = (
        result_df
        .sort_values(
            "C",
            ascending=False
        )
        .copy()
    )

    plot_df["plot_C"] = (
        plot_df["C"]
        .map(
            lambda x: f"{x:g}"
        )
    )

    # --------------------------------------------------------
    # AUC
    # --------------------------------------------------------

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        plot_df["plot_C"],
        plot_df["mean_test_AUC"],
        marker="o"
    )

    plt.xlabel(
        "Regression C Parameter"
    )

    plt.ylabel(
        "Mean Test AUC"
    )

    plt.title(
        "Logistic Regression C Cross-Validation — AUC"
    )

    plt.grid()

    plt.tight_layout()

    auc_path = data_set_path.replace(
        ".csv",
        "_crossval_regression_auc.png"
    )

    plt.savefig(
        auc_path
    )

    plt.close()


    # --------------------------------------------------------
    # LIFT
    # --------------------------------------------------------

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        plot_df["plot_C"],
        plot_df["mean_test_lift"],
        marker="o"
    )

    plt.xlabel(
        "Regression C Parameter"
    )

    plt.ylabel(
        "Mean Test Top-Decile Lift"
    )

    plt.title(
        "Logistic Regression C Cross-Validation — Lift"
    )

    plt.grid()

    plt.tight_layout()

    lift_path = data_set_path.replace(
        ".csv",
        "_crossval_regression_lift.png"
    )

    plt.savefig(
        lift_path
    )

    plt.close()


    # --------------------------------------------------------
    # NON-ZERO WEIGHTS
    # --------------------------------------------------------

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        plot_df["plot_C"],
        plot_df["n_weight"],
        marker="o"
    )

    plt.xlabel(
        "Regression C Parameter"
    )

    plt.ylabel(
        "Number of Non-Zero Weights"
    )

    plt.title(
        "Logistic Regression C — Model Complexity"
    )

    plt.grid()

    plt.tight_layout()

    weight_path = data_set_path.replace(
        ".csv",
        "_crossval_regression_weights.png"
    )

    plt.savefig(
        weight_path
    )

    plt.close()

    print(
        f"\nSaved AUC plot to:\n{auc_path}"
    )

    print(
        f"\nSaved lift plot to:\n{lift_path}"
    )

    print(
        f"\nSaved weight plot to:\n{weight_path}"
    )


# ============================================================
# LISTING 9.5 — C PARAMETER CROSS-VALIDATION
# ============================================================

def crossvalidate_regression(
    data_set_path,
    n_test_split=5
):

    # ========================================================
    # LOAD DATA
    # ========================================================

    X, y, dates = prepare_data(
        data_set_path
    )

    print(
        f"Observations: {len(X):,}"
    )

    print(
        f"Features: {X.shape[1]}"
    )

    print(
        f"First observation date: "
        f"{dates.min().date()}"
    )

    print(
        f"Last observation date: "
        f"{dates.max().date()}"
    )

    print(
        f"Overall churn rate: "
        f"{y.mean():.3%}"
    )


    # ========================================================
    # C VALUES FROM BOOK
    # ========================================================

    C_values = [
        0.64,
        0.32,
        0.16,
        0.08,
        0.04,
        0.02,
        0.01,
        0.005,
        0.0025
    ]


    # ========================================================
    # CREATE TEMPORAL FOLDS
    # ========================================================

    splits = create_date_splits(
        dates,
        n_test_split
    )


    # ========================================================
    # CROSS-VALIDATE EACH C
    # ========================================================

    all_results = []

    fold_results = []

    for c_value in C_values:

        print(
            f"\nTesting C = {c_value:g}"
        )

        auc_scores = []
        lift_scores = []

        for (
            fold_number,
            train_idx,
            test_idx,
            test_start,
            test_end
        ) in splits:

            X_train = X.iloc[
                train_idx
            ]

            y_train = y.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_test = y.iloc[
                test_idx
            ]

            # ------------------------------------------------
            # Skip training folds containing one class
            # ------------------------------------------------

            if y_train.nunique() < 2:

                print(
                    f"  Fold {fold_number}: skipped "
                    "(training data contains one class)"
                )

                fold_results.append(
                    {
                        "C":
                            c_value,

                        "fold":
                            fold_number,

                        "status":
                            "skipped_train_one_class",

                        "train_n":
                            len(X_train),

                        "test_n":
                            len(X_test),

                        "train_churn_rate":
                            y_train.mean(),

                        "test_churn_rate":
                            y_test.mean(),

                        "AUC":
                            np.nan,

                        "lift":
                            np.nan
                    }
                )

                continue


            # =================================================
            # L1 LOGISTIC REGRESSION
            # =================================================

            model = LogisticRegression(
                C=c_value,
                solver="saga",
                l1_ratio=1.0,
                fit_intercept=True,
                max_iter=5000,
                random_state=42
            )

            model.fit(
                X_train,
                y_train
            )


            # =================================================
            # CHURN PROBABILITY
            # =================================================

            class_to_index = {
                class_value: i
                for i, class_value
                in enumerate(
                    model.classes_
                )
            }

            churn_probability = (
                model.predict_proba(
                    X_test
                )[
                    :,
                    class_to_index[True]
                ]
            )


            # =================================================
            # AUC
            # =================================================

            if y_test.nunique() == 2:

                auc_score = (
                    roc_auc_score(
                        y_test,
                        churn_probability
                    )
                )

            else:

                auc_score = np.nan


            # =================================================
            # LIFT
            # =================================================

            lift_score = calc_lift(
                y_test,
                churn_probability
            )


            if np.isfinite(
                auc_score
            ):
                auc_scores.append(
                    auc_score
                )

            if np.isfinite(
                lift_score
            ):
                lift_scores.append(
                    lift_score
                )


            fold_results.append(
                {
                    "C":
                        c_value,

                    "fold":
                        fold_number,

                    "status":
                        "ok",

                    "train_n":
                        len(X_train),

                    "test_n":
                        len(X_test),

                    "train_start":
                        dates[
                            train_idx
                        ].min(),

                    "train_end":
                        dates[
                            train_idx
                        ].max(),

                    "test_start":
                        test_start,

                    "test_end":
                        test_end,

                    "train_churn_rate":
                        y_train.mean(),

                    "test_churn_rate":
                        y_test.mean(),

                    "AUC":
                        auc_score,

                    "lift":
                        lift_score
                }
            )


        # ====================================================
        # AGGREGATE RESULTS FOR THIS C
        # ====================================================

        all_results.append(
            {
                "C":
                    c_value,

                "mean_test_AUC":
                    np.mean(
                        auc_scores
                    )
                    if auc_scores
                    else np.nan,

                "std_test_AUC":
                    np.std(
                        auc_scores,
                        ddof=1
                    )
                    if len(auc_scores) > 1
                    else np.nan,

                "mean_test_lift":
                    np.mean(
                        lift_scores
                    )
                    if lift_scores
                    else np.nan,

                "std_test_lift":
                    np.std(
                        lift_scores,
                        ddof=1
                    )
                    if len(lift_scores) > 1
                    else np.nan,

                "valid_AUC_folds":
                    len(
                        auc_scores
                    ),

                "valid_lift_folds":
                    len(
                        lift_scores
                    )
            }
        )


    # ========================================================
    # MAIN RESULTS
    # ========================================================

    result_df = pd.DataFrame(
        all_results
    )


    # ========================================================
    # NON-ZERO WEIGHTS ON FULL HISTORICAL DATA
    # ========================================================

    weight_df = test_n_weights(
        X,
        y,
        C_values
    )

    result_df = result_df.merge(
        weight_df,
        on="C",
        how="left"
    )


    # ========================================================
    # SAVE MAIN CROSS-VALIDATION RESULTS
    # ========================================================

    crossval_path = (
        data_set_path.replace(
            ".csv",
            "_crossval.csv"
        )
    )

    result_df.to_csv(
        crossval_path,
        index=False
    )


    # ========================================================
    # SAVE FOLD-LEVEL RESULTS
    # ========================================================

    fold_df = pd.DataFrame(
        fold_results
    )

    fold_path = (
        data_set_path.replace(
            ".csv",
            "_crossval_folds.csv"
        )
    )

    fold_df.to_csv(
        fold_path,
        index=False
    )


    # ========================================================
    # DISPLAY
    # ========================================================

    print(
        f"\nSaved cross-validation results to:\n"
        f"{crossval_path}"
    )

    print(
        f"\nSaved fold results to:\n"
        f"{fold_path}"
    )

    print(
        "\nCross-validation summary:"
    )

    display(
        result_df
        .sort_values(
            "C",
            ascending=False
        )
    )


    # ========================================================
    # BEST C BY AUC
    # ========================================================

    valid_auc = result_df.dropna(
        subset=[
            "mean_test_AUC"
        ]
    )

    if not valid_auc.empty:

        best_auc_row = (
            valid_auc.loc[
                valid_auc[
                    "mean_test_AUC"
                ].idxmax()
            ]
        )

        print(
            "\nBest C by mean AUC:"
        )

        print(
            f"C = "
            f"{best_auc_row['C']:g}"
        )

        print(
            f"Mean AUC = "
            f"{best_auc_row['mean_test_AUC']:.4f}"
        )

        print(
            f"Mean lift = "
            f"{best_auc_row['mean_test_lift']:.4f}"
        )

        print(
            f"Non-zero weights = "
            f"{int(best_auc_row['n_weight'])}"
        )


    # ========================================================
    # BEST C BY LIFT
    # ========================================================

    valid_lift = result_df.dropna(
        subset=[
            "mean_test_lift"
        ]
    )

    if not valid_lift.empty:

        best_lift_row = (
            valid_lift.loc[
                valid_lift[
                    "mean_test_lift"
                ].idxmax()
            ]
        )

        print(
            "\nBest C by mean lift:"
        )

        print(
            f"C = "
            f"{best_lift_row['C']:g}"
        )

        print(
            f"Mean lift = "
            f"{best_lift_row['mean_test_lift']:.4f}"
        )

        print(
            f"Mean AUC = "
            f"{best_lift_row['mean_test_AUC']:.4f}"
        )

        print(
            f"Non-zero weights = "
            f"{int(best_lift_row['n_weight'])}"
        )


    # ========================================================
    # PLOTS
    # ========================================================

    plot_regression_test(
        data_set_path,
        result_df
    )


    return (
        result_df,
        fold_df
    )


# ============================================================
# RUN LISTING 9.5
# ============================================================

(
    crossval_results,
    crossval_fold_results
) = crossvalidate_regression(
    data_set_path=data_set_path,
    n_test_split=5
)

Observations: 32,897
Features: 2
First observation date: 2020-02-09
Last observation date: 2020-05-10
Overall churn rate: 1.961%

Testing C = 0.64
  Fold 1: skipped (training data contains one class)

Testing C = 0.32
  Fold 1: skipped (training data contains one class)

Testing C = 0.16
  Fold 1: skipped (training data contains one class)

Testing C = 0.08
  Fold 1: skipped (training data contains one class)

Testing C = 0.04
  Fold 1: skipped (training data contains one class)

Testing C = 0.02
  Fold 1: skipped (training data contains one class)

Testing C = 0.01
  Fold 1: skipped (training data contains one class)

Testing C = 0.005
  Fold 1: skipped (training data contains one class)

Testing C = 0.0025
  Fold 1: skipped (training data contains one class)

Saved cross-validation results to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval.csv

Saved fold results to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval_folds.csv

Cross-

,C,mean_test_AUC,std_test_AUC,mean_test_lift,std_test_lift,valid_AUC_folds,valid_lift_folds,n_weight
0,0.6400,0.569591,0.082650,2.046505,0.706558,4,4,2
1,0.3200,0.570673,0.082476,2.103914,0.742062,4,4,2
2,0.1600,0.572473,0.081855,2.103914,0.742062,4,4,2
3,0.0800,0.575949,0.080866,2.116867,0.744193,4,4,2
4,0.0400,0.574834,0.085500,1.887610,1.031789,4,4,2
5,0.0200,0.604193,0.076140,2.192392,0.862680,4,4,2
6,0.0100,0.614249,0.081085,2.360021,0.940625,4,4,2
7,0.0050,0.574666,0.090493,1.979044,0.943635,4,4,1
8,0.0025,0.500000,0.000000,1.456654,0.744692,4,4,0



Best C by mean AUC:
C = 0.01
Mean AUC = 0.6142
Mean lift = 2.3600
Non-zero weights = 2

Best C by mean lift:
C = 0.01
Mean lift = 2.3600
Mean AUC = 0.6142
Non-zero weights = 2

Saved AUC plot to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval_regression_auc.png

Saved lift plot to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval_regression_lift.png

Saved weight plot to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval_regression_weights.png


# XGBoost cross-validation

In [10]:
# ============================================================
# TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        values = set(series.dropna().unique())

        if not values.issubset({0, 1, 0.0, 1.0}):
            raise ValueError(
                f"Unexpected churn values: {values}"
            )

        return series.astype(int).astype(bool)

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = set(cleaned.unique()) - set(mapping.keys())

    if unknown:
        raise ValueError(
            f"Unknown is_churn values: {unknown}"
        )

    return cleaned.map(mapping).astype(bool)


# ============================================================
# PREPARE RAW / UNGROUPED DATA
#
# Book uses ext='' here.
# This means churn_dataset.csv rather than groupscore.
# ============================================================

def prepare_data(
    data_set_path,
    as_retention=False
):

    if not os.path.isfile(data_set_path):
        raise FileNotFoundError(
            f"Dataset not found:\n{data_set_path}"
        )

    data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    if "is_churn" not in data.columns:
        raise ValueError(
            '"is_churn" column is missing.'
        )

    is_churn = convert_churn_target(
        data["is_churn"]
    )

    if as_retention:
        y = ~is_churn
    else:
        y = is_churn

    X = data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():
        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            "NaN/non-numeric features found in:\n"
            f"{bad_columns}"
        )

    dates = pd.to_datetime(
        X.index.get_level_values(1),
        errors="raise"
    )

    # Chronological sort
    order = np.argsort(
        dates.values,
        kind="stable"
    )

    X = X.iloc[order].copy()
    y = y.iloc[order].copy()
    dates = dates[order]

    return X, y, dates


# ============================================================
# TOP-DECILE LIFT
# ============================================================

def calc_lift(
    y_true,
    y_pred,
    top_fraction=0.10
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    if len(y_true) == 0:
        return np.nan

    overall_churn = y_true.mean()

    if overall_churn <= 0:
        return np.nan

    order = np.argsort(
        y_pred
    )[::-1]

    y_sorted = y_true[
        order
    ]

    top_n = max(
        1,
        int(
            np.ceil(
                len(y_true) * top_fraction
            )
        )
    )

    top_churn_rate = (
        y_sorted[:top_n].mean()
    )

    return (
        top_churn_rate
        / overall_churn
    )


# ============================================================
# CREATE WHOLE-DATE TEMPORAL SPLITS
# ============================================================

def create_date_splits(
    dates,
    n_test_split
):

    unique_dates = np.array(
        sorted(
            pd.unique(dates)
        )
    )

    if len(unique_dates) < 3:
        raise ValueError(
            "Not enough unique dates for backtesting."
        )

    if n_test_split >= len(unique_dates):
        raise ValueError(
            "n_test_split must be smaller than "
            "the number of unique observation dates."
        )

    test_blocks = np.array_split(
        unique_dates[1:],
        n_test_split
    )

    splits = []

    for fold_number, test_dates in enumerate(
        test_blocks,
        start=1
    ):

        if len(test_dates) == 0:
            continue

        test_start = pd.Timestamp(
            test_dates[0]
        )

        test_end = pd.Timestamp(
            test_dates[-1]
        )

        train_idx = np.where(
            dates < test_start
        )[0]

        test_idx = np.where(
            (dates >= test_start)
            &
            (dates <= test_end)
        )[0]

        splits.append(
            (
                fold_number,
                train_idx,
                test_idx,
                test_start,
                test_end
            )
        )

    return splits


# ============================================================
# LISTING 9.6 — XGBOOST CROSS-VALIDATION
# ============================================================

def crossvalidate_xgb(
    data_set_path,
    n_test_split=5
):

    # --------------------------------------------------------
    # Load raw historical features
    # --------------------------------------------------------

    X, y, dates = prepare_data(
        data_set_path,
        as_retention=False
    )

    print(
        f"Observations: {len(X):,}"
    )

    print(
        f"Features: {X.shape[1]}"
    )

    print(
        f"First observation date: "
        f"{dates.min().date()}"
    )

    print(
        f"Last observation date: "
        f"{dates.max().date()}"
    )

    print(
        f"Overall churn rate: "
        f"{y.mean():.3%}"
    )

    # --------------------------------------------------------
    # Book parameter grid
    # --------------------------------------------------------

    test_params = {
        "max_depth": [
            1,
            2,
            4,
            6
        ],

        "learning_rate": [
            0.1,
            0.2,
            0.3,
            0.4
        ],

        "n_estimators": [
            20,
            40,
            80,
            120
        ],

        "min_child_weight": [
            3,
            6,
            9,
            12
        ]
    }

    # 4 × 4 × 4 × 4 = 256 combinations
    param_combinations = []

    for max_depth in test_params["max_depth"]:
        for learning_rate in test_params["learning_rate"]:
            for n_estimators in test_params["n_estimators"]:
                for min_child_weight in test_params["min_child_weight"]:

                    param_combinations.append(
                        {
                            "max_depth":
                                max_depth,

                            "learning_rate":
                                learning_rate,

                            "n_estimators":
                                n_estimators,

                            "min_child_weight":
                                min_child_weight
                        }
                    )

    print(
        f"\nParameter combinations: "
        f"{len(param_combinations)}"
    )

    # --------------------------------------------------------
    # Temporal folds
    # --------------------------------------------------------

    splits = create_date_splits(
        dates,
        n_test_split
    )

    all_results = []

    best_auc = -np.inf
    best_params = None

    # ========================================================
    # GRID SEARCH
    # ========================================================

    for combo_number, params in enumerate(
        param_combinations,
        start=1
    ):

        auc_scores = []
        lift_scores = []

        for (
            fold_number,
            train_idx,
            test_idx,
            test_start,
            test_end
        ) in splits:

            X_train = X.iloc[
                train_idx
            ]

            y_train = y.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]

            y_test = y.iloc[
                test_idx
            ]

            # Skip folds where training contains one class
            if y_train.nunique() < 2:
                continue

            model = xgb.XGBClassifier(
                objective="binary:logistic",

                max_depth=params[
                    "max_depth"
                ],

                learning_rate=params[
                    "learning_rate"
                ],

                n_estimators=params[
                    "n_estimators"
                ],

                min_child_weight=params[
                    "min_child_weight"
                ],

                eval_metric="logloss",

                random_state=42,

                n_jobs=-1
            )

            model.fit(
                X_train,
                y_train.astype(int)
            )

            churn_prob = (
                model.predict_proba(
                    X_test
                )[:, 1]
            )

            # --------------------------------------------
            # AUC
            # --------------------------------------------

            if y_test.nunique() == 2:

                auc = roc_auc_score(
                    y_test,
                    churn_prob
                )

                auc_scores.append(
                    auc
                )

            # --------------------------------------------
            # Lift
            # --------------------------------------------

            lift = calc_lift(
                y_test,
                churn_prob
            )

            if np.isfinite(lift):

                lift_scores.append(
                    lift
                )

        # ====================================================
        # Aggregate combination
        # ====================================================

        mean_auc = (
            np.mean(auc_scores)
            if auc_scores
            else np.nan
        )

        mean_lift = (
            np.mean(lift_scores)
            if lift_scores
            else np.nan
        )

        std_auc = (
            np.std(
                auc_scores,
                ddof=1
            )
            if len(auc_scores) > 1
            else np.nan
        )

        std_lift = (
            np.std(
                lift_scores,
                ddof=1
            )
            if len(lift_scores) > 1
            else np.nan
        )

        row = {
            **params,

            "mean_test_AUC":
                mean_auc,

            "std_test_AUC":
                std_auc,

            "mean_test_lift":
                mean_lift,

            "std_test_lift":
                std_lift,

            "valid_AUC_folds":
                len(auc_scores),

            "valid_lift_folds":
                len(lift_scores)
        }

        all_results.append(
            row
        )

        if (
            np.isfinite(mean_auc)
            and mean_auc > best_auc
        ):

            best_auc = mean_auc
            best_params = params.copy()

        if (
            combo_number % 25 == 0
            or combo_number == len(param_combinations)
        ):

            print(
                f"Tested "
                f"{combo_number}/"
                f"{len(param_combinations)} "
                f"parameter combinations"
            )

    # ========================================================
    # RESULTS
    # ========================================================

    result_df = pd.DataFrame(
        all_results
    )

    result_df = (
        result_df
        .sort_values(
            "mean_test_AUC",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            "_crossval_xgb.csv"
        )
    )

    result_df.to_csv(
        save_path,
        index=False
    )

    print(
        f"\nSaved test scores to:\n"
        f"{save_path}"
    )

    print(
        "\nTop XGBoost parameter combinations:"
    )

    display(
        result_df.head(20)
    )

    if best_params is None:

        raise ValueError(
            "No valid XGBoost parameter combination "
            "produced an AUC score."
        )

    # ========================================================
    # REFIT BEST MODEL ON FULL HISTORICAL DATA
    # ========================================================

    print(
        "\nBest parameters by temporal CV AUC:"
    )

    print(
        best_params
    )

    print(
        f"\nBest mean temporal AUC: "
        f"{best_auc:.4f}"
    )

    best_model = xgb.XGBClassifier(
        objective="binary:logistic",

        max_depth=
            best_params[
                "max_depth"
            ],

        learning_rate=
            best_params[
                "learning_rate"
            ],

        n_estimators=
            best_params[
                "n_estimators"
            ],

        min_child_weight=
            best_params[
                "min_child_weight"
            ],

        eval_metric="logloss",

        random_state=42,

        n_jobs=-1
    )

    best_model.fit(
        X,
        y.astype(int)
    )

    # ========================================================
    # SAVE MODEL + FEATURE ORDER
    # ========================================================

    pickle_path = (
        data_set_path.replace(
            ".csv",
            "_xgb_model.pkl"
        )
    )

    model_package = {
        "model":
            best_model,

        "feature_names":
            list(X.columns),

        "target":
            "churn",

        "best_params":
            best_params,

        "mean_cv_auc":
            best_auc
    }

    with open(
        pickle_path,
        "wb"
    ) as fid:

        pickle.dump(
            model_package,
            fid
        )

    print(
        f"\nSaved XGBoost model to:\n"
        f"{pickle_path}"
    )

    return (
        result_df,
        best_model,
        best_params
    )


# ============================================================
# RUN LISTING 9.6
# ============================================================

(
    xgb_crossval_results,
    best_xgb_model,
    best_xgb_params
) = crossvalidate_xgb(
    data_set_path=data_set_path,
    n_test_split=5
)

Observations: 32,897
Features: 9
First observation date: 2020-02-09
Last observation date: 2020-05-10
Overall churn rate: 1.961%

Parameter combinations: 256
Tested 25/256 parameter combinations
Tested 50/256 parameter combinations
Tested 75/256 parameter combinations
Tested 100/256 parameter combinations
Tested 125/256 parameter combinations
Tested 150/256 parameter combinations
Tested 175/256 parameter combinations
Tested 200/256 parameter combinations
Tested 225/256 parameter combinations
Tested 250/256 parameter combinations
Tested 256/256 parameter combinations

Saved test scores to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_crossval_xgb.csv

Top XGBoost parameter combinations:


,max_depth,learning_rate,n_estimators,min_child_weight,mean_test_AUC,std_test_AUC,mean_test_lift,std_test_lift,valid_AUC_folds,valid_lift_folds
0,1,0.1,80,3,0.579628,0.058591,2.247484,0.471652,4,4
1,1,0.1,120,3,0.578900,0.056787,2.232245,0.482672,4,4
2,1,0.2,40,3,0.578699,0.057106,2.227214,0.442445,4,4
3,1,0.2,80,6,0.578670,0.054074,2.218779,0.452144,4,4
4,1,0.3,80,6,0.578582,0.045245,2.184868,0.514683,4,4
5,1,0.2,20,12,0.578571,0.047391,2.065991,0.467090,4,4
6,1,0.2,40,6,0.578274,0.060405,2.379815,0.365405,4,4
7,1,0.4,40,6,0.578251,0.052710,2.198395,0.470474,4,4
8,1,0.4,40,12,0.578204,0.058190,2.264362,0.353553,4,4
9,1,0.3,40,3,0.578196,0.057088,2.278986,0.567510,4,4



Best parameters by temporal CV AUC:
{'max_depth': 1, 'learning_rate': 0.1, 'n_estimators': 80, 'min_child_weight': 3}

Best mean temporal AUC: 0.5796

Saved XGBoost model to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_xgb_model.pkl
